# 🎫 Event Entry Pass Verification System
NumPy array + QR code generation/decoding + grant/deny logic, in one notebook (Colab-ready).

In [ ]:
!pip install -q qrcode[pil] pyzbar
!apt-get -qq install -y libzbar0

In [ ]:
import numpy as np
import qrcode
from PIL import Image
from pyzbar.pyzbar import decode as qr_decode

# 1. NumPy structured array = the participant database
PARTICIPANT_DTYPE = np.dtype([
    ('participant_id', 'U10'), ('name', 'U40'), ('ticket_type', 'U10'),
    ('registered', 'bool'), ('checked_in', 'bool'),
])

records = np.array([
    ('EVT001', 'Rekha Priya', 'VIP', True, False),
    ('EVT002', 'Arun Kumar', 'General', True, False),
    ('EVT003', 'Divya S', 'General', True, False),
    ('EVT005', 'Meena Loganathan', 'General', False, False),
], dtype=PARTICIPANT_DTYPE)

records

In [ ]:
# 2. Generate a QR pass for each participant
qr_images = {}
for row in records:
    pid = str(row['participant_id'])
    img = qrcode.make(pid)
    qr_images[pid] = img
    img.save(f'{pid}.png')

qr_images['EVT001']

In [ ]:
# 3. Scan (decode) a QR pass
def scan_qr(image: Image.Image) -> str | None:
    result = qr_decode(image)
    return result[0].data.decode('utf-8') if result else None

scanned_id = scan_qr(Image.open('EVT002.png'))
print('Scanned ID:', scanned_id)

In [ ]:
# 4 & 5. Retrieve the record from the NumPy array and display / grant-deny
def verify_entry(records, participant_id):
    mask = records['participant_id'] == participant_id
    matches = records[mask]
    if matches.size == 0:
        return records, f"\u274c Access Denied - '{participant_id}' not found (Invalid ID)"
    rec = matches[0]
    if not rec['registered']:
        return records, f"\u274c Access Denied - {rec['name']} is not a confirmed registration"
    if rec['checked_in']:
        return records, f"\u26a0\ufe0f Duplicate Entry - {rec['name']} already checked in"
    records['checked_in'][mask] = True
    return records, f"\u2705 Entry Granted - Welcome {rec['name']} ({rec['ticket_type']})"

records, message = verify_entry(records, scanned_id)
print(message)

In [ ]:
# 6. Handle invalid / unregistered / duplicate cases gracefully
for test_id in ['EVT002', 'EVT005', 'EVT999']:
    records, message = verify_entry(records, test_id)
    print(test_id, '->', message)

## Bonus: quick analytics with Pandas + Matplotlib

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.DataFrame(records)
display(df)

df['checked_in'].value_counts().plot(kind='bar', title='Check-in status')
plt.show()

The full interactive version of this project (upload/camera QR scanning, live grant/deny UI, CSV export, analytics dashboard) is deployed as a Streamlit app - see the README / deployment link.